# Playground Series S6E4 — Predicting Irrigation Need

**Task**: Multi-class classification (Low / Medium / High)  
**Metric**: Balanced Accuracy  
**Key challenge**: Severe class imbalance — "High" is only 3.3% of training data

In [ ]:
# Install missing packages (Kaggle environment)
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

try:
    import optuna
except ImportError:
    pip_install('optuna')
    import optuna

print('Environment ready')

In [ ]:
import os
import subprocess
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── GPU detection ─────────────────────────────────────────────────────────────
def _has_gpu():
    try:
        r = subprocess.run(['nvidia-smi'], capture_output=True, timeout=5)
        return r.returncode == 0
    except Exception:
        return False

USE_GPU = _has_gpu()
print(f'GPU available: {USE_GPU}')
if USE_GPU:
    subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=False)

# ── Global constants ──────────────────────────────────────────────────────────
RANDOM_STATE = 42
N_FOLDS      = 5
np.random.seed(RANDOM_STATE)

TARGET = 'Irrigation_Need'

CAT_COLS = [
    'Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season',
    'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region'
]
NUM_COLS = [
    'Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity',
    'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours',
    'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm'
]

print(f'Numerical features  : {len(NUM_COLS)}')
print(f'Categorical features: {len(CAT_COLS)}')

In [ ]:
import os

# Kaggle can mount competition data at different paths — try both
_candidates = [
    '/kaggle/input/playground-series-s6e4',
    '/kaggle/input/competitions/playground-series-s6e4',
]
KAGGLE_INPUT = next((p for p in _candidates if os.path.exists(p)), None)
LOCAL_INPUT  = 'datasets'

DATA_DIR = KAGGLE_INPUT if KAGGLE_INPUT else LOCAL_INPUT

# Diagnostics — helps debug path issues
print(f'Data directory   : {DATA_DIR}')
print(f'/kaggle/input/   : {os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else "n/a"}')

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')
sub   = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

print(f'Train shape      : {train.shape}')
print(f'Test  shape      : {test.shape}')
print(f'\nTarget distribution:')
print(train[TARGET].value_counts(normalize=True).round(4))

## Step 2 — Exploratory Data Analysis

Before any modelling we inspect:
- **Missing values** — are imputation strategies needed?
- **Class balance** — quantifies the imbalance problem we need to solve
- **Feature distributions per class** — reveals which signals separate Low / Medium / High
- **Categorical cardinality** — informs encoding choices
- **Correlation heatmap** — detects multicollinearity among numeric features

In [ ]:
# ── Data quality checks ──────────────────────────────────────────────────────
print('Missing values in train :', train.isnull().sum().sum())
print('Missing values in test  :', test.isnull().sum().sum())
print('\nColumn dtypes:')
print(train.dtypes)
print('\nCategorical feature cardinality:')
for col in CAT_COLS:
    print(f'  {col}: {train[col].nunique()} unique → {sorted(train[col].unique())[:5]}')

In [ ]:
# ── Statistical summary of numerical features ────────────────────────────────
print('Numerical feature statistics:')
train[NUM_COLS].describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

In [ ]:
# ── Class imbalance — bar chart ───────────────────────────────────────────────
# "High" irrigation need is severely under-represented (~3.3%).
# This drives our choice to use class_weight='balanced' in all models.
palette = {'Low': '#4C9BE8', 'Medium': '#F5A623', 'High': '#E84C4C'}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = train[TARGET].value_counts().reindex(['Low', 'Medium', 'High'])
axes[0].bar(counts.index, counts.values, color=[palette[c] for c in counts.index], edgecolor='white')
axes[0].set_title('Class Distribution (absolute counts)', fontsize=13)
axes[0].set_ylabel('Count')
for bar, val in zip(axes[0].patches, counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
                 f'{val:,}', ha='center', fontsize=10)

pcts = counts / counts.sum() * 100
axes[1].pie(pcts.values, labels=pcts.index, autopct='%1.1f%%',
            colors=[palette[c] for c in pcts.index], startangle=140)
axes[1].set_title('Class Distribution (%)', fontsize=13)

plt.suptitle('Target Variable: Irrigation Need', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
print('Key insight: "High" class is heavily imbalanced — requires balanced weighting.')

In [ ]:
# ── Numerical feature distributions per class ─────────────────────────────────
# Overlapping histograms show which features best separate the three classes.
# Clear separation → strong predictive signal for that feature.
fig, axes = plt.subplots(3, 4, figsize=(18, 11))
axes = axes.flatten()

class_order = ['Low', 'Medium', 'High']
colors_cls  = {'Low': '#4C9BE8', 'Medium': '#F5A623', 'High': '#E84C4C'}

for i, col in enumerate(NUM_COLS):
    for cls in class_order:
        axes[i].hist(
            train.loc[train[TARGET] == cls, col],
            bins=40, alpha=0.55, density=True,
            label=cls, color=colors_cls[cls]
        )
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].legend(fontsize=7)
    axes[i].set_xlabel('')

# Hide the unused 12th subplot (11 features, 12 subplots)
for j in range(len(NUM_COLS), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numerical Feature Distributions by Irrigation Need Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('Features with clear class separation are the most valuable predictors.')

## Step 3 — Feature Engineering

We construct domain-inspired features that directly encode agronomic stress signals:

| Feature | Intuition |
|---------|-----------|
| `Water_Stress` | Temperature / Soil Moisture — high value = crops need more water |
| `Evapotranspiration_Proxy` | Sunlight × Temp × (1 − Humidity) — approximates ET₀ |
| `Moisture_Deficit` | Gap between field capacity (65%) and current moisture |
| `Total_Water_Input` | Rainfall + Previous Irrigation — total water already available |
| `Rain_vs_Prev` | Ratio of natural vs. applied water |
| `Soil_Quality` | Organic Carbon / EC — captures soil health |

All features are ratio/interaction terms — they stay meaningful across different scales.

In [ ]:
def engineer_features(df):
    """Add domain-specific interaction and ratio features."""
    df = df.copy()

    # Water availability & stress signals
    df['Water_Stress']              = df['Temperature_C'] / (df['Soil_Moisture'] + 1e-3)
    df['Rain_vs_Prev']              = df['Rainfall_mm'] / (df['Previous_Irrigation_mm'] + 1e-3)
    df['Moisture_Deficit']          = 65 - df['Soil_Moisture']           # distance from field capacity
    df['Total_Water_Input']         = df['Rainfall_mm'] + df['Previous_Irrigation_mm']
    df['Prev_Irrig_Moisture_ratio'] = df['Previous_Irrigation_mm'] / (df['Soil_Moisture'] + 1e-3)

    # Atmospheric demand (approximated evapotranspiration)
    df['Temp_Humidity']              = df['Temperature_C'] * (1 - df['Humidity'] / 100)
    df['Evapotranspiration_Proxy']   = df['Sunlight_Hours'] * df['Temp_Humidity']
    df['Wind_Evap']                  = df['Wind_Speed_kmh'] * df['Temperature_C'] / (df['Humidity'] + 1)
    df['Rain_per_hour']              = df['Rainfall_mm'] / (df['Sunlight_Hours'] + 1)

    # Soil quality interactions
    df['EC_pH_interaction'] = df['Electrical_Conductivity'] * df['Soil_pH']
    df['Carbon_Moisture']   = df['Organic_Carbon'] * df['Soil_Moisture']
    df['Soil_Quality']      = df['Organic_Carbon'] / (df['Electrical_Conductivity'] + 1e-3)

    return df

train = engineer_features(train)
test  = engineer_features(test)

NEW_FEATURES = [
    'Water_Stress', 'Rain_vs_Prev', 'Temp_Humidity', 'Evapotranspiration_Proxy',
    'Moisture_Deficit', 'EC_pH_interaction', 'Carbon_Moisture', 'Wind_Evap',
    'Rain_per_hour', 'Prev_Irrig_Moisture_ratio', 'Total_Water_Input', 'Soil_Quality'
]
FEATURE_COLS = NUM_COLS + NEW_FEATURES + CAT_COLS

print(f'Total features: {len(FEATURE_COLS)}  '
      f'({len(NUM_COLS)} raw num + {len(NEW_FEATURES)} engineered + {len(CAT_COLS)} cat)')

In [ ]:
# ── Target encoding & categorical label encoding ──────────────────────────────
# Map string labels → integers so sklearn metrics work correctly.
# Categoricals are integer-encoded here for LightGBM/XGBoost;
# CatBoost will receive the original string columns via its own Pool interface.
target_map = {'Low': 0, 'Medium': 1, 'High': 2}
target_inv = {v: k for k, v in target_map.items()}

y      = train[TARGET].map(target_map).values
X      = train[FEATURE_COLS].copy()
X_test = test[FEATURE_COLS].copy()

# Fit LabelEncoder on train+test combined to avoid unseen-category errors
for col in CAT_COLS:
    le = LabelEncoder()
    combined = pd.concat([X[col], X_test[col]], axis=0)
    le.fit(combined)
    X[col]      = le.transform(X[col])
    X_test[col] = le.transform(X_test[col])

print(f'X shape     : {X.shape}')
print(f'y shape     : {y.shape}')
print(f'Class counts: {dict(zip(target_map.keys(), np.bincount(y)))}')

## Step 4 — Cross-Validation Setup

We use **Stratified 5-Fold CV** to preserve class proportions in every fold.
Out-of-fold (OOF) predictions serve as a proxy for the public leaderboard score
and are used to find optimal ensemble weights without data leakage.

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# OOF probability arrays — shape (n_train, 3)
oof_lgbm = np.zeros((len(X), 3))
oof_xgb  = np.zeros((len(X), 3))
oof_cat  = np.zeros((len(X), 3))

# Averaged test probability arrays — shape (n_test, 3)
test_lgbm = np.zeros((len(X_test), 3))
test_xgb  = np.zeros((len(X_test), 3))
test_cat  = np.zeros((len(X_test), 3))

X_arr      = X.values
X_test_arr = X_test.values

print(f'Cross-validation: {N_FOLDS}-fold stratified')

## Step 5 — LightGBM

In [ ]:
lgbm_params = {
    'objective':         'multiclass',
    'num_class':         3,
    'metric':            'multi_logloss',
    'n_estimators':      1200,
    'learning_rate':     0.04,
    'num_leaves':        127,
    'max_depth':         -1,
    'min_child_samples': 50,
    'subsample':         0.8,
    'subsample_freq':    1,
    'colsample_bytree':  0.8,
    'reg_alpha':         0.1,
    'reg_lambda':        1.0,
    'class_weight':      'balanced',
    'random_state':      RANDOM_STATE,
    'n_jobs':            -1,
    'verbose':           -1,
}

lgbm_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_arr, y)):
    X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
    y_tr, y_va = y[tr_idx],     y[va_idx]

    model = lgb.LGBMClassifier(**lgbm_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)]
    )

    oof_lgbm[va_idx] = model.predict_proba(X_va)
    test_lgbm        += model.predict_proba(X_test_arr) / N_FOLDS

    score = balanced_accuracy_score(y_va, oof_lgbm[va_idx].argmax(1))
    lgbm_scores.append(score)
    print(f'  Fold {fold+1}: {score:.5f}  (best iter: {model.best_iteration_})')

lgbm_cv = balanced_accuracy_score(y, oof_lgbm.argmax(1))
print(f'\nLightGBM OOF Balanced Accuracy: {lgbm_cv:.5f}')

In [ ]:
# GPU speeds up LightGBM training ~3-5x on Kaggle T4
lgbm_params = {
    'objective':         'multiclass',
    'num_class':         3,
    'metric':            'multi_logloss',
    'n_estimators':      1200,
    'learning_rate':     0.04,
    'num_leaves':        127,
    'max_depth':         -1,
    'min_child_samples': 50,
    'subsample':         0.8,
    'subsample_freq':    1,
    'colsample_bytree':  0.8,
    'reg_alpha':         0.1,
    'reg_lambda':        1.0,
    'class_weight':      'balanced',
    'random_state':      RANDOM_STATE,
    'n_jobs':            -1,
    'verbose':           -1,
    # GPU: switch from CPU histogram to GPU-accelerated tree building
    **({'device': 'gpu'} if USE_GPU else {}),
}

lgbm_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_arr, y)):
    X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
    y_tr, y_va = y[tr_idx],     y[va_idx]

    model = lgb.LGBMClassifier(**lgbm_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)]
    )

    oof_lgbm[va_idx] = model.predict_proba(X_va)
    test_lgbm        += model.predict_proba(X_test_arr) / N_FOLDS

    score = balanced_accuracy_score(y_va, oof_lgbm[va_idx].argmax(1))
    lgbm_scores.append(score)
    print(f'  Fold {fold+1}: {score:.5f}  (best iter: {model.best_iteration_})')

lgbm_cv = balanced_accuracy_score(y, oof_lgbm.argmax(1))
print(f'\nLightGBM OOF Balanced Accuracy: {lgbm_cv:.5f}')

In [ ]:
# Per-sample class weights compensate for imbalance inside XGBoost's loss
sample_weights = compute_sample_weight('balanced', y)

xgb_params = {
    'objective':        'multi:softprob',
    'num_class':        3,
    'eval_metric':      'mlogloss',
    'n_estimators':     1200,
    'learning_rate':    0.04,
    'max_depth':        7,
    'min_child_weight': 10,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'reg_alpha':        0.1,
    'reg_lambda':       1.0,
    'random_state':     RANDOM_STATE,
    'n_jobs':           -1,
    'tree_method':      'hist',
    'verbosity':        0,
}

xgb_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_arr, y)):
    X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
    y_tr, y_va = y[tr_idx],     y[va_idx]
    sw_tr      = sample_weights[tr_idx]

    model = xgb.XGBClassifier(**xgb_params, early_stopping_rounds=100)
    model.fit(X_tr, y_tr, sample_weight=sw_tr,
              eval_set=[(X_va, y_va)], verbose=False)

    oof_xgb[va_idx] = model.predict_proba(X_va)
    test_xgb        += model.predict_proba(X_test_arr) / N_FOLDS

    score = balanced_accuracy_score(y_va, oof_xgb[va_idx].argmax(1))
    xgb_scores.append(score)
    print(f'  Fold {fold+1}: {score:.5f}')

xgb_cv = balanced_accuracy_score(y, oof_xgb.argmax(1))
print(f'\nXGBoost OOF Balanced Accuracy: {xgb_cv:.5f}')

In [ ]:
sample_weights = compute_sample_weight('balanced', y)

xgb_params = {
    'objective':        'multi:softprob',
    'num_class':        3,
    'eval_metric':      'mlogloss',
    'n_estimators':     1200,
    'learning_rate':    0.04,
    'max_depth':        7,
    'min_child_weight': 10,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'reg_alpha':        0.1,
    'reg_lambda':       1.0,
    'random_state':     RANDOM_STATE,
    'n_jobs':           -1,
    # GPU: 'device=cuda' replaces tree_method='gpu_hist' in XGBoost ≥ 2.0
    'device':           'cuda' if USE_GPU else 'cpu',
    'tree_method':      'hist',
    'verbosity':        0,
}

xgb_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_arr, y)):
    X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
    y_tr, y_va = y[tr_idx],     y[va_idx]
    sw_tr      = sample_weights[tr_idx]

    model = xgb.XGBClassifier(**xgb_params, early_stopping_rounds=100)
    model.fit(X_tr, y_tr, sample_weight=sw_tr,
              eval_set=[(X_va, y_va)], verbose=False)

    oof_xgb[va_idx] = model.predict_proba(X_va)
    test_xgb        += model.predict_proba(X_test_arr) / N_FOLDS

    score = balanced_accuracy_score(y_va, oof_xgb[va_idx].argmax(1))
    xgb_scores.append(score)
    print(f'  Fold {fold+1}: {score:.5f}')

xgb_cv = balanced_accuracy_score(y, oof_xgb.argmax(1))
print(f'\nXGBoost OOF Balanced Accuracy: {xgb_cv:.5f}')

In [ ]:
from catboost import Pool as CatPool

# CatBoost receives raw string categoricals — no label encoding needed
X_cat_str      = train[FEATURE_COLS].copy()
X_test_cat_str = test[FEATURE_COLS].copy()

cat_params = {
    'iterations':            1200,
    'learning_rate':         0.04,
    'depth':                 7,
    'l2_leaf_reg':           3.0,
    'bagging_temperature':   0.5,
    'random_strength':       1.0,
    'auto_class_weights':    'Balanced',
    'loss_function':         'MultiClass',
    'eval_metric':           'Accuracy',
    'early_stopping_rounds': 100,
    'random_seed':           RANDOM_STATE,
    'verbose':               0,
    'task_type':             'CPU',
}

cat_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_cat_str, y)):
    X_tr_df = X_cat_str.iloc[tr_idx]
    X_va_df = X_cat_str.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    train_pool = CatPool(X_tr_df, label=y_tr, cat_features=CAT_COLS)
    val_pool   = CatPool(X_va_df, label=y_va, cat_features=CAT_COLS)
    test_pool  = CatPool(X_test_cat_str,      cat_features=CAT_COLS)

    model = CatBoostClassifier(**cat_params)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)

    oof_cat[va_idx] = model.predict_proba(val_pool)
    test_cat        += model.predict_proba(test_pool) / N_FOLDS

    score = balanced_accuracy_score(y_va, oof_cat[va_idx].argmax(1))
    cat_scores.append(score)
    print(f'  Fold {fold+1}: {score:.5f}')

cat_cv = balanced_accuracy_score(y, oof_cat.argmax(1))
print(f'\nCatBoost OOF Balanced Accuracy: {cat_cv:.5f}')

In [ ]:
from catboost import Pool as CatPool

X_cat_str      = train[FEATURE_COLS].copy()
X_test_cat_str = test[FEATURE_COLS].copy()

cat_params = {
    'iterations':            1200,
    'learning_rate':         0.04,
    'depth':                 7,
    'l2_leaf_reg':           3.0,
    'bagging_temperature':   0.5,
    'random_strength':       1.0,
    'auto_class_weights':    'Balanced',
    'loss_function':         'MultiClass',
    'eval_metric':           'Accuracy',
    'early_stopping_rounds': 100,
    'random_seed':           RANDOM_STATE,
    'verbose':               0,
    # GPU: CatBoost gets the biggest speedup (~10x) — recommended to always use on Kaggle
    'task_type':             'GPU' if USE_GPU else 'CPU',
}

cat_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_cat_str, y)):
    X_tr_df = X_cat_str.iloc[tr_idx]
    X_va_df = X_cat_str.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    train_pool = CatPool(X_tr_df, label=y_tr, cat_features=CAT_COLS)
    val_pool   = CatPool(X_va_df, label=y_va, cat_features=CAT_COLS)
    test_pool  = CatPool(X_test_cat_str,      cat_features=CAT_COLS)

    model = CatBoostClassifier(**cat_params)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)

    oof_cat[va_idx] = model.predict_proba(val_pool)
    test_cat        += model.predict_proba(test_pool) / N_FOLDS

    score = balanced_accuracy_score(y_va, oof_cat[va_idx].argmax(1))
    cat_scores.append(score)
    print(f'  Fold {fold+1}: {score:.5f}')

cat_cv = balanced_accuracy_score(y, oof_cat.argmax(1))
print(f'\nCatBoost OOF Balanced Accuracy: {cat_cv:.5f}')

In [ ]:
scores_summary = pd.DataFrame({
    'Model':            ['LightGBM', 'XGBoost', 'CatBoost'],
    'OOF_Balanced_Acc': [lgbm_cv,    xgb_cv,    cat_cv],
}).sort_values('OOF_Balanced_Acc', ascending=False)

print(scores_summary.to_string(index=False))
best_model_name = scores_summary.iloc[0]['Model']
print(f'\nBest base model: {best_model_name} → will tune with Optuna')

## Step 9 — Optuna Hyperparameter Tuning (Best Model)

In [ ]:
skf3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

def lgbm_objective(trial):
    params = {
        'objective':         'multiclass',
        'num_class':         3,
        'metric':            'multi_logloss',
        'n_estimators':      trial.suggest_int('n_estimators', 600, 2000),
        'learning_rate':     trial.suggest_float('learning_rate', 0.02, 0.1, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 63, 255),
        'max_depth':         trial.suggest_int('max_depth', 6, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'subsample_freq':    1,
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_split_gain':    trial.suggest_float('min_split_gain', 0.0, 0.5),
        'class_weight':      'balanced',
        'random_state':      RANDOM_STATE,
        'n_jobs':            -1,
        'verbose':           -1,
    }
    oof_preds = np.zeros((len(X_arr), 3))
    for tr_idx, va_idx in skf3.split(X_arr, y):
        X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
        y_tr, y_va = y[tr_idx],     y[va_idx]
        m = lgb.LGBMClassifier(**params)
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(80, verbose=False), lgb.log_evaluation(-1)])
        oof_preds[va_idx] = m.predict_proba(X_va)
    return balanced_accuracy_score(y, oof_preds.argmax(1))

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study.optimize(lgbm_objective, n_trials=50, show_progress_bar=True)

print(f'\nBest Optuna trial score: {study.best_value:.5f}')
print('Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

## Step 10 — Retrain Tuned LightGBM with 5-Fold CV

In [ ]:
best_params = study.best_params.copy()
best_params.update({
    'objective':      'multiclass',
    'num_class':      3,
    'metric':         'multi_logloss',
    'class_weight':   'balanced',
    'subsample_freq': 1,
    'random_state':   RANDOM_STATE,
    'n_jobs':         -1,
    'verbose':        -1,
})

oof_tuned    = np.zeros((len(X_arr), 3))
test_tuned   = np.zeros((len(X_test_arr), 3))
tuned_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_arr, y)):
    X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
    y_tr, y_va = y[tr_idx],     y[va_idx]

    m = lgb.LGBMClassifier(**best_params)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
          callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])

    oof_tuned[va_idx] = m.predict_proba(X_va)
    test_tuned        += m.predict_proba(X_test_arr) / N_FOLDS

    score = balanced_accuracy_score(y_va, oof_tuned[va_idx].argmax(1))
    tuned_scores.append(score)
    print(f'  Fold {fold+1}: {score:.5f}  (best iter: {m.best_iteration_})')

tuned_cv = balanced_accuracy_score(y, oof_tuned.argmax(1))
print(f'\nTuned LightGBM OOF Balanced Accuracy: {tuned_cv:.5f}')

## Step 11 — Ensemble

In [ ]:
from itertools import product as iproduct

best_ens_score = 0
best_weights   = (0.4, 0.2, 0.2, 0.2)

# Grid search over simplex {w1+w2+w3+w4 = 1, all ≥ 0} with step 0.1
steps = np.arange(0, 1.01, 0.1)
for w1, w2, w3 in iproduct(steps, steps, steps):
    w4 = round(1.0 - w1 - w2 - w3, 5)
    if w4 < 0:
        continue
    ens = w1 * oof_lgbm + w2 * oof_xgb + w3 * oof_cat + w4 * oof_tuned
    s   = balanced_accuracy_score(y, ens.argmax(1))
    if s > best_ens_score:
        best_ens_score = s
        best_weights   = (w1, w2, w3, w4)

w1, w2, w3, w4 = best_weights
print(f'Best weights  : LGBM={w1:.1f}  XGB={w2:.1f}  CAT={w3:.1f}  LGBM_tuned={w4:.1f}')
print(f'Ensemble OOF  : {best_ens_score:.5f}')

# Final blended test predictions
test_ensemble = w1 * test_lgbm + w2 * test_xgb + w3 * test_cat + w4 * test_tuned

## Step 12 — Final Score Summary

In [ ]:
final_summary = pd.DataFrame({
    'Model': ['LightGBM (base)', 'XGBoost', 'CatBoost', 'LightGBM (tuned)', 'Ensemble'],
    'OOF_Balanced_Accuracy': [lgbm_cv, xgb_cv, cat_cv, tuned_cv, best_ens_score],
}).sort_values('OOF_Balanced_Accuracy', ascending=False).reset_index(drop=True)

print('=== Final CV Score Summary ===')
print(final_summary.to_string(index=False))

In [ ]:
# ── Choose best prediction source & save ─────────────────────────────────────
if best_ens_score >= tuned_cv:
    final_preds_proba = test_ensemble
    chosen = 'Ensemble'
else:
    final_preds_proba = test_tuned
    chosen = 'Tuned LightGBM'

print(f'Chosen predictions: {chosen}')

final_preds_int    = final_preds_proba.argmax(1)
final_preds_labels = [target_inv[i] for i in final_preds_int]

submission = pd.DataFrame({
    'id':              test['id'].values,
    'Irrigation_Need': final_preds_labels
})

OUT_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
submission.to_csv(f'{OUT_DIR}/submission.csv', index=False)
print(f'Submission saved : {OUT_DIR}/submission.csv  ({len(submission)} rows)')
print('\nPrediction distribution:')
print(submission['Irrigation_Need'].value_counts())
submission.head()

In [ ]:
# Use best predictions (ensemble or tuned, whichever is higher)
if best_ens_score >= tuned_cv:
    final_preds_proba = test_ensemble
    chosen = 'Ensemble'
else:
    final_preds_proba = test_tuned
    chosen = 'Tuned LightGBM'

print(f'Chosen predictions: {chosen}')

final_preds_int    = final_preds_proba.argmax(1)
final_preds_labels = [target_inv[i] for i in final_preds_int]

submission = pd.DataFrame({
    'id':             test['id'].values,
    'Irrigation_Need': final_preds_labels
})

submission.to_csv('submission.csv', index=False)
print(f'Submission saved: submission.csv ({len(submission)} rows)')
print('\nPrediction distribution:')
print(submission['Irrigation_Need'].value_counts())
submission.head()